# Convolutional Neural Networks for Galaxy Image Classification

## Learning Objectives
By completing this tutorial, you will learn how to:
- **Load and preprocess** galaxy cutout images from the Euclid survey dataset
- **Build a Convolutional Neural Network (CNN)** using TensorFlow/Keras
- **Train and validate** a deep learning model on galaxy morphology classification
- **Evaluate model performance** using multiple metrics (accuracy, loss, confusion matrix, ROC-AUC)
- **Interpret results** and understand how CNNs extract features from images

## Project Overview
This tutorial uses real galaxy images from the Euclid Q1 dataset to classify galaxies into two morphological types:
- **Elliptical Galaxies (E)**: Smooth, featureless appearance (Class 0)
- **Disk Galaxies (D)**: Structured, with visible disk and spiral features (Class 1)

You will build a CNN model that automatically learns to distinguish these galaxy types from images alone, without hand-crafted features.

## Dataset
**Source:** Euclid Q1 Galaxy Sample  
**Size:** 10,000 galaxy images at 256×256 pixels (RGB)  
**Labels:** Binary classification (Elliptical vs Disk)  
**Download:** [Euclid Q1 Dataset](https://drive.google.com/file/d/1EVW0o0_UmvIEVOgZPQQxbIkOX2NvF6Nq/view?usp=sharing)

## Tutorial Structure
1. **Setup:** Import libraries and configure paths
2. **Data Loading:** Load images and explore dataset properties
3. **Dataset Creation:** Prepare TensorFlow datasets with batching and shuffling
4. **Visualization:** Inspect sample images from each class
5. **Model Architecture:** Build a CNN with convolutional and dense layers
6. **Training:** Train the model with callbacks for early stopping and learning rate reduction
7. **Evaluation:** Assess performance on test set with various metrics

# 1. Import Libraries and Setup

### Testing TensorFlow Installation
Run the cell below to verify TensorFlow is properly installed and working:

In [ ]:
import tensorflow as tf
import numpy as np

def predict(data):
    a = tf.keras.Sequential([tf.keras.layers.Dense(4, input_shape=(16,))])

    return a.predict(data)

fake_data = np.zeros((100, 16))

print(fake_data.shape)
# Works
for i in range(4):
    print(predict(fake_data).shape)

In [ ]:
# Import required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## Configuration: File Paths

We define the paths to our data files. 

**Key Points:**
- `images_dir`: Directory containing the 64×64 pixel galaxy cutout images (JPEG format)
- `catalogue`: CSV file with metadata for each galaxy (morphology, measurements, etc.)

Make sure these paths match your local file structure!

In [ ]:
# Directory containing galaxy cutout images
images_dir = ""

# Catalogue path
catalogue = ""

## Utility Functions

### The `create_dataset()` Function

**Purpose:** Convert image file paths and labels into optimized TensorFlow datasets

**Key Features:**
1. **Lazy Loading:** Images loaded only when batches are created (saves memory)
2. **Parallel Processing:** Multiple images loaded simultaneously for speed
3. **Shuffling:** Training data randomized to improve model generalization
4. **Normalization:** Pixel values scaled to [0, 1] range for training stability
5. **Batching:** Data grouped into batches for efficient GPU processing
6. **Prefetching:** Next batch prepared while current batch trains

**Parameters:**
- `image_paths`: Array of file paths to images
- `labels`: Class labels (0 = Elliptical, 1 = Disk)
- `batch_size`: Images per batch (default: 64)
- `img_size`: Target size after resizing (default: 128×128)
- `shuffle`: Whether to randomize sample order

In [ ]:
def create_dataset(image_paths, labels, batch_size=64, img_size=(128, 128), shuffle=True):
    """
    Create a TensorFlow dataset from image paths and labels with robust JPEG handling.
    """
    def load_image(path, label):
        # Load and decode JPEG with error handling
        image = tf.io.read_file(path)
        try:
            image = tf.image.decode_jpeg(image, channels=3)
        except:
            # Return a placeholder if decode fails
            image = tf.zeros((*img_size, 3), dtype=tf.uint8)
        
        # Ensure image is uint8
        image = tf.cast(image, tf.uint8)
        
        # Resize to target size
        image = tf.image.resize(image, img_size)
        
        # Normalize to [0, 1] range
        image = tf.cast(image, tf.float32) / 255.0
        
        return image, label
    
    # Create dataset from tensor slices
    dataset = tf.data.Dataset.from_tensor_slices((image_paths.flatten(), labels))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(image_paths))
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

## 2. Load and Explore the Data

Exploration is the first step of any machine learning project. In this section, we:
- Load the CSV catalogue containing galaxy metadata
- Discover available galaxy images in the dataset
- Verify data integrity and understand class distribution
- Create absolute file paths to ensure robust image loading

### Exploration Steps:
1. Display CSV structure and column names
2. Count total available images
3. Verify paths to ensure all images can be accessed
4. Check class distribution (are classes balanced?)
5. Create binary labels based on galaxy morphology

In [ ]:
# Read catalogue

# Display the first few rows of the catalogue


In [ ]:
# Check available images
image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
print(f"Total images available: {len(image_files)}")
print(f"Sample image filenames:")
for img_file in image_files[:5]:
    print(f"  - {img_file}")

In [ ]:
# Sample only part of the data

In [ ]:
# Filter to only samples with available images

In [ ]:
# Create binary labels from galaxy_class (E vs D)
# E (Elliptical) = 0, D (Disk) = 1

# Convert relative paths to absolute paths

# Verify paths exist

# Keep only rows with valid paths

## 3. Data Splitting and TensorFlow Dataset Creation

### Train/Validation/Test Split Strategy

#### Why Three Sets?
- **Training Set (60%):** Updates model weights during training
- **Validation Set (25%):** Tunes hyperparameters and detects overfitting
- **Test Set (15%):** Final held-out set for unbiased performance evaluation

#### Why Stratify?
**Stratified splitting** ensures each subset maintains the same class proportions as the original dataset. This prevents:
- Imbalanced training (e.g., mostly Elliptical galaxies)
- Misleading validation/test performance
- Biased model predictions

### TensorFlow Dataset Benefits
Instead of loading all images into memory (which could be GB), TensorFlow Datasets:
1. **Load on-demand:** Only load images needed for current batch
2. **Process in parallel:** Multiple images loaded simultaneously
3. **Enable prefetching:** Next batch loaded while current batch trains
4. **Support shuffling:** Randomize sample order for better training

**Result:** Faster training and lower memory usage!

In [ ]:
# Split into training (60%), validation (25%), and testing (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    ,
    ,
    test_size=0.15,
    random_state=42,
    stratify=
)

X_train, X_val, y_train, y_val = train_test_split(
    ,
    ,
    test_size=0.3,  # ~25% of total
    random_state=42,
    stratify=
)

print(f"\nDataset splits:")
print(f"  Training set: {len(X_train)}")
print(f"  Validation set: {len(X_val)}")
print(f"  Test set: {len(X_test)}")

# Create TensorFlow datasets
train_dataset = create_dataset(X_train, y_train, batch_size=64, img_size=(64, 64), shuffle=True)
val_dataset = create_dataset(X_val, y_val, batch_size=64, img_size=(64, 64), shuffle=False)
test_dataset = create_dataset(X_test, y_test, batch_size=64, img_size=(64, 64), shuffle=False)

In [ ]:
# Verify data loading - sample a batch from training data
print("Testing data loading from training dataset...")
try:
    # Get one batch
    for images, labels in train_dataset.take(1):
        print(f"Batch images shape: {images.shape}")
        print(f"Batch labels shape: {labels.shape}")
        print(f"Image value range: [{images.numpy().min():.3f}, {images.numpy().max():.3f}]")
        print(f"Label distribution in batch: {np.bincount(labels.numpy().flatten().astype(int))}")
        
        # Check if any images are blank/corrupted
        image_means = tf.reduce_mean(images, axis=[1, 2, 3]).numpy()
        print(f"Image intensity means: min={image_means.min():.3f}, max={image_means.max():.3f}, mean={image_means.mean():.3f}")
        
        if (image_means == 0).any():
            print("WARNING: Some images have zero mean - possible loading errors!")
        
        print("\nData loading test passed - images appear valid!")
except Exception as e:
    print(f"ERROR loading data: {e}")
    import traceback
    traceback.print_exc()

## 4. Visualizing Sample Galaxy Images

### What to Look For

**Elliptical Galaxies (E):**
- Smooth, featureless appearance
- Approximately circular or elongated shape
- Little visible structure
- Examples: smooth, symmetric profiles

**Disk Galaxies (D):**
- Visible disk structure
- Often spiral or bar-like features
- Complex morphology
- Examples: spiral arms, clear disks

In [ ]:
# Visualize sample images from the training set with their labels
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Load and display 10 random validation images
for i, (images, labels) in enumerate(train_dataset.take(4)):
    for j in range(min(10, images.shape[0])):
        ax = axes[j // 5, j % 5]
        
        # Display image (denormalize from [0, 1] to [0, 255])
        image_array = (images[j].numpy() * 255).astype('uint8')
        ax.imshow(image_array)
        
        label = labels[j].numpy()
        class_name = 'Disk (D)' if label == 1 else 'Elliptical (E)'
        ax.set_title(f'{class_name}')
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from the validation set with their labels
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Load and display 10 random validation images
for i, (images, labels) in enumerate(val_dataset.take(4)):
    for j in range(min(10, images.shape[0])):
        ax = axes[j // 5, j % 5]
        
        # Display image (denormalize from [0, 1] to [0, 255])
        image_array = (images[j].numpy() * 255).astype('uint8')
        ax.imshow(image_array)
        
        label = labels[j].numpy()
        class_name = 'Disk (D)' if label == 1 else 'Elliptical (E)'
        ax.set_title(f'{class_name}')
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Grab one batch from each pipeline and compare raw ranges
for x_train_batch, y_train_batch in train_dataset.take(1):
    for x_val_batch, y_val_batch in val_dataset.take(1):
        print("Train batch range:", tf.reduce_min(x_train_batch).numpy(), tf.reduce_max(x_train_batch).numpy())
        print("Val batch range:  ", tf.reduce_min(x_val_batch).numpy(), tf.reduce_max(x_val_batch).numpy())
        break
    break

## 5. Building the CNN Model Architecture

### What You'll Learn
Convolutional Neural Networks use hierarchical feature extraction:
- **Conv2D layers** extract local spatial patterns (edges, textures, shapes)
- **MaxPooling** reduces spatial dimensions while preserving important features
- **Dense layers** learn high-level patterns from extracted features
- **Dropout** prevents overfitting by randomly deactivating neurons during training

### Model Architecture Breakdown

#### Convolutional Blocks
Each block contains:
1. **Conv2D(filters, kernel_size):** Applies filters to extract features
   - Filters learn to detect patterns (edges, corners, textures)
   - Kernel size (e.g., 5×5) determines the receptive field size
2. **BatchNormalization:** Normalizes layer outputs for training stability
3. **MaxPooling2D:** Downsamples feature maps (e.g., 7×7 pooling reduces dimensions)
4. **Dropout(rate):** Randomly disable neurons to prevent overfitting

#### Fully Connected Layers
After flattening, dense layers learn complex decision boundaries:
- **First dense layer:** Combines all extracted features
- **Second dense layer:** Further abstracts high-level patterns
- **Output layer:** Single sigmoid unit outputs probability [0, 1]
  - Sigmoid ensures output is interpretable as probability
  - Values close to 0 → "Elliptical" (Class 0)
  - Values close to 1 → "Disk" (Class 1)

### Why These Choices?
- **Feature Hierarchy:** Early layers detect simple features (edges), later layers detect complex patterns (galaxy shapes)
- **Regularization:** Dropout and BatchNorm prevent overfitting on limited data
- **Binary Classification:** Sigmoid activation with 1 output unit is standard for 2-class problems

In [ ]:
def create_cnn_model(input_shape=(128, 128, 3)):
    """
    Create a CNN model for galaxy classification.
    
    Args:
        input_shape: Shape of input images (height, width, channels)
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(64, (5, 5), activation='relu', input_shape=input_shape, name='conv1'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((7, 7), name='maxpool1'),
        layers.Dropout(0.3),
        
        # Flatten
        layers.Flatten(name='flatten'),
        
        # Fully Connected Layers
        layers.Dense(32, activation='relu', name='dense1'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(16, activation='relu', name='dense2'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Output Layer
        layers.Dense(1, activation='sigmoid', name='output')
    ])

    return model

In [ ]:
# Create the model
model = create_cnn_model(input_shape=(64, 64, 3))

# Compile the model
model.compile(
    optimizer=,
    loss=,
    metrics=
)

# Display model summary
model.summary()

## 6. Training the Model

### Training Callbacks Explained

#### Early Stopping
```
EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
```
**Purpose:** Stop training when validation performance plateaus
- **monitor='val_loss':** Watch validation loss (loss on unseen data)
- **patience=5:** Wait 5 epochs without improvement before stopping
- **restore_best_weights=True:** Revert to best model, not final model

**Why It Helps:** Prevents overfitting (model memorizing training data while forgetting to generalize)

#### Learning Rate Reduction
```
ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
```
**Purpose:** Reduce learning rate when training plateaus
- **factor=0.5:** Multiply learning rate by 0.5 when stuck
- **patience=5:** Wait 5 epochs without improvement

**Why It Helps:** Fine-tunes weights when large steps aren't helping anymore

### Optimizer: Adam
- Adapts learning rate for each parameter automatically
- Works well with deep learning and image data
- Standard choice for CNN training

### Loss Function: Binary Crossentropy
- Measures difference between predicted probabilities and true labels
- Standard for binary classification
- Lower loss = better predictions

### Training Progress Interpretation:
- **Good:** Both training and validation loss decrease together
- **Overfitting:** Training loss decreases but validation loss increases
- **Problem:** Losses don't decrease (learning rate too high or model too simple)

In [ ]:
# Define callbacks for training
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
preds = model.predict(val_dataset)
print("Prediction stats:", preds.min(), preds.max(), preds.mean())
print("Prediction std:", preds.std())
print("Label distribution:", np.unique(y_val, return_counts=True))

## 7. Analyzing Training History

### How to Interpret the Plots

#### Healthy Training Curves
- Both training and validation loss **decrease together**
- Both metrics **smooth** (not noisy)
- Gap between curves is **small** (good generalization)

#### Signs of Overfitting
- Training loss continues decreasing
- Validation loss **increases or plateaus**
- Growing gap between training and validation curves
- **Solution:** Add dropout, reduce model size, or use more data

#### Signs of Underfitting
- Both curves decrease very slowly
- High validation loss after many epochs
- **Solution:** Increase model capacity, train longer, or improve data quality

### Analysis Questions
1. When does validation loss stop improving?
2. Is there a large gap between training and validation performance?
3. At what epoch does early stopping trigger?
4. Overall, did the model learn effectively?

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss Over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Model Evaluation and Performance Analysis

### Key Evaluation Metrics Explained

#### Accuracy
- **Definition:** (Correct Predictions) / (Total Predictions)
- **Limitation:** Misleading if classes are imbalanced
- **Example:** 95% accuracy could mean just predicting the majority class!

#### Precision & Recall (Per Class)
- **Precision:** Of galaxies predicted as "Disk", how many actually are "Disk"?
  - Formula: True Positives / (True Positives + False Positives)
  - High precision = few false alarms
- **Recall:** Of actual "Disk" galaxies, how many did we find?
  - Formula: True Positives / (True Positives + False Negatives)
  - High recall = few missed detections
- **Trade-off:** Precision and recall usually trade off (can't maximize both)

#### Confusion Matrix
|                  | Predicted E | Predicted D |
|------------------|------------|------------|
| **Actual E**     | TN         | FP         |
| **Actual D**     | FN         | TP         |

- **Diagonal elements (TN, TP):** Correct predictions
- **Off-diagonal (FP, FN):** Incorrect predictions
- **What to check:** Are any off-diagonal values surprisingly high?

#### ROC-AUC (Receiver Operating Characteristic - Area Under Curve)
- **Meaning:** Probability that model ranks a random positive higher than random negative
- **Range:** 0 to 1 (higher is better)
- **Interpretation:**
  - 0.5 = Random guessing (useless)
  - 0.7-0.8 = Good discrimination
  - 0.9+ = Excellent discrimination
  - 1.0 = Perfect predictions (rare)

### Evaluation Strategy
1. **Check accuracy** as overall baseline
2. **Examine precision/recall** for each class
3. **Visualize confusion matrix** for detailed error patterns
4. **Calculate ROC-AUC** for classification ability
5. **Draw conclusions** about model reliability

### Critical Questions
- Is the model performing equally well on both classes?
- Are there systematic errors (e.g., confusing Disks for Ellipticals)?
- Is performance good enough for scientific use?
- How would you improve the model if needed?

In [ ]:
# Evaluate model on test set
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Get predictions on test set
test_predictions = model.predict(test_dataset)
test_pred_labels = (test_predictions > 0.5).astype(int).flatten()